# Feature Engineering

This notebook focuses on transforming telemetry data into actionable performance indicators. By calculating metrics for speed, braking, throttle application, and gear usage, the process creates a structured dataset that quantifies driving behavior, enabling a deeper understanding of track management.

## Setup and Dependencies

Load dependencies, configure the notebook path, and prepare the modules used in the exploration steps.

In [1]:
import sys
import fastf1
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.config import *
from src.load_save_data import *
from src.preprocess_data import *
from src.explore_data import *
from src.feature_engineering import *

## Load Processed Data

Fetch cleaned and interpolated telemetry data for both teammates. Loading pre-prepared data ensures that feature calculations are performed on standardized, high-quality inputs.

In [2]:
session = fastf1.get_session(YEAR, GRAND_PRIX, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, TEAM)
segment = get_segment(session, driver_1, driver_2)
turns = get_turns(session)

try:
    lap_1, lap_2 = load_cache(YEAR, GRAND_PRIX, segment, driver_1, driver_2)
except Exception as e:
    print(
        f"Processed data for {YEAR} {GRAND_PRIX} {driver_1} and {driver_2} not found: {e}"
    )

req         WARNING 	DEFAULT CACHE ENABLED! (99.92 MB) C:\Users\joseg\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '6', '44', '23', '87', '10', '55', '14', '30', '22', '27', '5', '31', '7', '18']



Processed data for 2025 Japanese Grand Prix ALO and STR found in cache.
Driver 1: ALO, Driver 2: STR, Segment: Q1


## Compute Segment Times

In [3]:
times_segments_1 = analyze_time_segments(lap_1, turns)
times_segments_2 = analyze_time_segments(lap_2, turns)

print(f"Times segments for {driver_1}:")
display(times_segments_1)
print(f"Times segments for {driver_2}:")
display(times_segments_2)

Times segments for ALO:


,Segment,SegmentTime
0,0,8.318758
1,1,1.748213
2,2,3.791294
3,3,1.917242
4,4,2.377794
5,5,2.454693
6,6,4.926145
7,7,5.348176
8,8,2.303449
9,9,5.639452


Times segments for STR:


,Segment,SegmentTime
0,0,8.393278
1,1,1.827904
2,2,3.792102
3,3,1.895474
4,4,2.404166
5,5,2.476878
6,6,5.019166
7,7,5.431299
8,8,2.299278
9,9,5.683200


## Compute Segments' Speed Features

Compute speed-related performance metrics such as: entry speed, minimum speed and exit speed throughout the corners. This analysis examines speed profiles relative to track turns to identify differences in corner entry and exit speeds, providing insights into raw pace and efficiency.

In [4]:
speed_segments_1 = analyze_speed_segments(lap_1, turns)
speed_segments_2 = analyze_speed_segments(lap_2, turns)

print(f"Speed segments for {driver_1}:")
display(speed_segments_1)
print(f"Speed segments for {driver_2}:")
display(speed_segments_2)

Speed segments for ALO:


,Segment,MeanSpeed,StdSpeed,MaximumSpeed,MinimumSpeed
0,0,308.654308,13.501688,323.000000,273.115577
1,1,220.167876,27.328718,265.373486,185.286291
2,2,212.064530,24.974962,248.688758,175.000000
3,3,248.365024,7.297717,254.930126,233.134308
4,4,234.063819,5.675690,242.826880,225.316284
5,5,228.876084,9.094881,239.000000,205.638172
6,6,226.394855,20.518257,251.836056,192.259037
7,7,277.354752,9.817394,288.000000,253.864366
8,8,230.114884,39.725882,267.891889,150.428652
9,9,214.515664,43.651110,263.760410,131.647637


Speed segments for STR:


,Segment,MeanSpeed,StdSpeed,MaximumSpeed,MinimumSpeed
0,0,306.627264,12.822320,321.872229,274.938637
1,1,219.329791,35.510813,285.476753,175.582293
2,2,211.133132,26.296725,249.637206,171.071110
3,3,247.429717,3.394529,252.946651,239.381784
4,4,232.776822,5.055776,241.534565,224.118413
5,5,225.683613,7.321945,237.389314,213.387221
6,6,222.315852,17.797864,248.827249,192.897075
7,7,273.440085,11.153524,287.000000,249.686738
8,8,231.965897,29.597502,260.737964,167.811698
9,9,210.603090,43.964242,263.000000,133.547327


## Compute Segments' Throttle Features
Calculate throttle usage segments. This analysis determines when and how aggressively each driver applies the throttle coming out of corners, serving as a critical indicator of traction management and power deployment. Each segment registers: mean throttle, throttle standard deviation and full throttle percentage. 

In [5]:
throttle_segments_1 = analyze_throttle_segments(lap_1, turns)
throttle_segments_2 = analyze_throttle_segments(lap_2, turns)

print(f"Throttle segments for {driver_1}:")
display(throttle_segments_1)
print(f"Throttle segments for {driver_2}:")
display(throttle_segments_2)

Throttle segments for ALO:


,Segment,MeanThrottle,StdThrottle,FullThrottlePercentage
0,0,92.647100,21.778178,86.904762
1,1,5.478588,9.032246,0.000000
2,2,89.058511,22.484614,77.777778
3,3,69.351708,18.929261,18.750000
4,4,72.245615,24.165788,36.842105
5,5,51.720823,26.911346,5.263158
6,6,86.000652,28.211506,72.972973
7,7,93.091439,18.227434,87.755102
8,8,50.805418,40.363838,17.647059
9,9,91.314393,22.835585,84.615385


Throttle segments for STR:


,Segment,MeanThrottle,StdThrottle,FullThrottlePercentage
0,0,98.293451,8.509816,97.619048
1,1,2.272783,5.640167,0.000000
2,2,92.352880,14.044979,77.777778
3,3,79.305871,17.203975,25.000000
4,4,67.921587,30.576488,26.315789
5,5,60.225034,43.590728,42.105263
6,6,82.506505,27.461524,62.162162
7,7,91.615371,25.066260,89.795918
8,8,55.102156,46.488676,35.294118
9,9,87.046364,28.912688,79.487179


## Compute Segments' Gear Shifts Features

Examine gear shift patterns across the lap registering: number of shifts, lowest gear, highest gear, minimum rpm, maximum rpm, mean rpn and rpm standard deviation. Comparing gear selection at different track segments reveals variations in driving technique, engine management, and how torque and traction are balanced.

In [6]:
gear_shifts_segments_1 = analyze_gear_shifts_segments(lap_1, turns)
gear_shifts_segments_2 = analyze_gear_shifts_segments(lap_2, turns)

print(f"Gear shifts segments for {driver_1}:")
display(gear_shifts_segments_1)
print(f"Gear shifts segments for {driver_2}:")
display(gear_shifts_segments_2)

Gear shifts segments for ALO:


,Segment,NumberOfShifts,LowestGear,HighestGear,MinimumRPM,MaximumRPM,MeanRPM,StdRPM
0,0,1,7,8,10342,11836,11178.976190,332.855661
1,1,3,5,8,9816,10659,10285.230769,253.066577
2,2,1,5,6,9541,11838,10901.555556,739.657838
3,3,0,6,6,10923,12004,11676.125000,369.464094
4,4,0,6,6,10766,11357,11071.789474,212.931021
5,5,1,5,6,10634,11467,11033.315789,205.927186
6,6,2,5,7,10356,11943,11074.621622,453.344017
7,7,0,7,7,10586,11776,11374.204082,365.357586
8,8,1,4,7,9756,10737,10343.647059,266.318639
9,9,5,3,7,9794,11655,10985.538462,433.235492


Gear shifts segments for STR:


,Segment,NumberOfShifts,LowestGear,HighestGear,MinimumRPM,MaximumRPM,MeanRPM,StdRPM
0,0,1,7,8,10244,11668,11116.130952,319.209633
1,1,3,5,8,9634,11120,10277.615385,461.640108
2,2,1,5,6,9402,11837,10839.222222,745.754669
3,3,0,6,6,11166,11914,11662.187500,233.172102
4,4,0,6,6,10444,11467,10979.789474,281.442431
5,5,1,5,6,10433,11503,10950.842105,305.265142
6,6,1,5,6,10497,11603,11086.135135,318.456360
7,7,2,6,7,10359,11755,11260.122449,429.054126
8,8,1,5,6,10071,12135,11343.764706,635.719821
9,9,4,4,7,8411,11464,10527.358974,841.791526


## Compute Turns' Time

In [7]:
time_turns_1 = analyze_time_turns(lap_1, turns)
time_turns_2 = analyze_time_turns(lap_2, turns)

print(f"Turns' time for {driver_1}:")
display(time_turns_1)
print(f"Turns' time for {driver_2}:")
display(time_turns_2)

Turns' time for ALO:


,Turn,TurnTime
0,1,1.251325
1,2,2.012193
2,3,1.487866
3,4,1.507092
4,5,1.593337
5,6,1.801300
6,7,1.331637
7,8,1.208773
8,9,2.723001
9,10,1.290761


Turns' time for STR:


,Turn,TurnTime
0,1,1.249592
1,2,2.072301
2,3,1.498084
3,4,1.456939
4,5,1.394205
5,6,1.653006
6,7,1.353549
7,8,1.229143
8,9,2.714242
9,10,1.312186


## Compute Turns' Braking Features

Identify and analyze braking zones throughout the lap. This section evaluates how each driver approaches braking points—specifically focusing on the start and end of braking events—to highlight differences in commitment and corner entry style registering information such as: braking duration, braking distance and braking point.

In [8]:
braking_turns_1 = analyze_braking_turns(lap_1, turns)
braking_turns_2 = analyze_braking_turns(lap_2, turns)

print(f"Braking in turns for {driver_1}:")
display(braking_turns_1)
print(f"Braking in turns for {driver_2}:")
display(braking_turns_2)

Braking in turns for ALO:


,Turn,BrakingPoint,BrakingDistance,BrakingDuration
0,1,31.944321,94.451661,1.403598
1,2,NaN,NaN,NaN
2,3,NaN,NaN,NaN
3,4,NaN,NaN,NaN
4,5,NaN,NaN,NaN
5,6,53.215447,42.932573,0.710531
6,7,NaN,NaN,NaN
7,8,33.348841,51.519088,0.718115
8,9,68.549995,68.692117,1.414791
9,10,NaN,NaN,NaN


Braking in turns for STR:


,Turn,BrakingPoint,BrakingDistance,BrakingDuration
0,1,31.944321,103.038175,1.571767
1,2,NaN,NaN,NaN
2,3,NaN,NaN,NaN
3,4,46.566067,42.932573,0.643467
4,5,33.166824,34.346058,0.559168
5,6,44.628932,8.586515,0.137719
6,7,NaN,NaN,NaN
7,8,33.348841,34.346058,0.476000
8,9,68.549995,77.278632,1.612747
9,10,NaN,NaN,NaN


## Compute Turns' Speed Features

In [9]:
speed_turns_1 = analyze_speed_turns(lap_1, turns)
speed_turns_2 = analyze_speed_turns(lap_2, turns)

print(f"Turns' speed for {driver_1}:")
display(speed_turns_1)
print(f"Turns' speed for {driver_2}:")
display(speed_turns_2)

Turns' speed for ALO:


,Turn,EntrySpeed,ApexSpeed,ExitSpeed
0,1,316.849927,226.000000,218.830361
1,2,218.830361,175.000000,188.211202
2,3,237.207326,239.046293,254.000000
3,4,249.991280,225.316284,229.824552
4,5,242.826880,228.000000,230.017651
5,6,239.000000,192.259037,193.285593
6,7,247.558435,248.333829,263.927344
7,8,287.241671,255.966174,253.581031
8,9,222.547171,131.647637,156.726907
9,10,259.496525,260.650977,244.285862


Turns' speed for STR:


,Turn,EntrySpeed,ApexSpeed,ExitSpeed
0,1,310.668174,221.783322,215.922341
1,2,215.922341,171.071110,183.305473
2,3,235.214762,238.790426,247.528850
3,4,246.000000,227.000000,228.707714
4,5,241.534565,218.000000,225.453623
5,6,235.749412,194.030461,192.897075
6,7,238.936092,240.479466,259.000000
7,8,287.000000,246.460688,247.619907
8,9,240.579802,133.547327,144.203072
9,10,256.376645,257.988499,246.919433


## Cache Processed Data

Integrate the newly calculated metrics into the driver data structures and save the augmented dataset to the cache. This ensures that extracted features are persistent and readily available for subsequent performance comparisons and visualization steps.

In [ ]:
segments_features_1 = build_segments_dataframe(times_segments_1, speed_segments_1, throttle_segments_1, gear_shifts_segments_1)
segments_features_2 = build_segments_dataframe(times_segments_2, speed_segments_2, throttle_segments_2, gear_shifts_segments_2)

turns_features_1 = build_turns_dataframe(time_turns_1, speed_turns_1, braking_turns_1)
turns_features_2 = build_turns_dataframe(time_turns_2, speed_turns_2, braking_turns_2)

lap_1["Segments"] = segments_features_1
lap_2["Segments"] = segments_features_2

lap_1["Turns"] = turns_features_1
lap_2["Turns"] = turns_features_2

print(f"Segments and Turns features for {driver_1}:")
display(lap_1["Segments"])
display(lap_1["Turns"])
print(f"Segments and Turns features for {driver_2}:")
display(lap_2["Segments"])
display(lap_2["Turns"])

save_dataframe(lap_1, "Segments", YEAR, GRAND_PRIX)
save_dataframe(lap_2, "Segments", YEAR, GRAND_PRIX)
save_dataframe(lap_1, "Turns", YEAR, GRAND_PRIX)
save_dataframe(lap_2, "Turns", YEAR, GRAND_PRIX)

try:
    save_cache(YEAR, GRAND_PRIX, segment, lap_1, lap_2)
except Exception as e:
    print(f"Failed to save lap cache: {e}")

TypeError: build_turns_dataframe() missing 1 required positional argument: 'braking'